# പാഠം 18 (തുടര്‍ത്തിരിക്കുന്നത്): ഒരു *മനുഷ്യൻ* പ്രവർത്തനം അനുവദിച്ചതായി തെളിയിക്കുന്ന രസീതുകൾ

ഈ പാഠം എന്ത് **ഏജന്റ്** ചെയ്തുവെന്നും എന്താണ് **ഗേറ്റ്** തീരുമാനിച്ചുവെന്നതുമാണ് തെളിയിക്കുന്നത്. ഈ നോട്ട്ബുക്കിൽ കാണാത്ത പകുതി ചേർക്കുന്നു: ഒരു **പേര് വന്ന മനുഷ്യന്** പ്രത്യേകിച്ച് അംഗീകരിച്ച **നിജമായ** നടപടി — പൂർണ്ണ കാനോണിക്കൽ പ്രവർത്തനത്തെക്കുറിച്ച് ഒരു വേറിട്ട, മനുഷ്യന് കൈവശമുള്ള ഒപ്പ്, ഒഫ്ലൈനിൽ പരിശോധിക്കപ്പെടുന്നു.

ഇവിടെ ഉള്ള രണ്ട് കലാലേഖനങ്ങളും **പാഠത്തിന്റെ രസീതുകളിലെതന്നെയുള്ള ഒരു നിർദ്ദിഷ്ട ആനുകൂല്യ രൂപം** ഉപയോഗിച്ചിരിക്കുന്നു: ഒരു സമതളമായ പയ്ലോഡ് `type` ഫീൽഡുമായി, Ed25519 ഉപയോഗിച്ച് നേരിട്ട് കാനോണിക്കൽ JCS ബൈറ്റുകളെ ഒപ്പുവെച്ചത്, ഒറ്റപ്പെട്ട `signature` ഒബ്‌ജക്റ്റ് അനുബന്ധവും (ഒപ്പുവെച്ച ബൈറ്റുകളിൽ നിന്ന് ഒഴിവാക്കപ്പെട്ടതും). അംഗീകൃത രസീത ഒരു പുതിയ `type` ആണ് (`human.approval.v1`), പ്രവർത്തന ടൈപ്പിനൊപ്പം, അതുകൊണ്ടു ഒരു `verify_chain` ഒരേ കോഡ് പാത്തിലായി രണ്ടു കലാലേഖന പ്രസ്‌കരണങ്ങൾക്കും പാടുദോഷം പൊതുകുന്നു. ഈ മനുഷ്യ അംഗീകൃത രസീത പഠനാനുകൂലമായ കൂട്ടിച്ചേരലാണ്, draft-farley-acta-signed-receipts നിർവചിച്ച രസീത തരം അല്ല.

പ്രധാന നോട്ട്ബുക്കിലെ ഡെമോ പരിശോധകനിൽ നിന്ന് ഒരുവിധമുള്ള ഉന്നതിവാണ്: ഇവിടെ `signature.key_id` **പിന്നുചേർന്ന കീ രജിസ്ട്രിയുമായി** പൊരുത്തപ്പെടുത്തുന്നു, രസീതയിലുള്ള പബ്ലിക് കീ വിഭ്രാന്തപ്പെടുത്താതെ. ഇതാണ് പാഠത്തിന്റെ തന്നെ ചെക്ക്ലിസ്റ്റ് ശിപാർശ ചെയ്യുന്ന ഉത്പാദന നിലപാട് ("പരിശോധന പബ്ലിക് കീ പ്രസിദ്ധീകരിക്കുക"), ഇത് നികത്തി ഒഴിവാക്കൽ അല്ലാതെ ഒറ്റപ്പെടലായ നിർബന്ധമായി മാറുന്നു.

ഈ നോട്ട്ബുക്ക് പഠിപ്പിക്കുന്നത്: **ഒപ്പുവച്ച അംഗീകാരം അതാത് അധികാരമല്ല.** അധികാരം നിലനിൽക്കുന്നത് ഒരേ കാനോണിക്കൽ പ്രവർത്തനത്തിൽ അംഗീകൃത രസീതയും പ്രവർത്തന രസീതയും ബന്ധപ്പെട്ടിരിക്കുമ്പോഴാണ്, ഇപ്പോഴും നിലവിലുള്ള നയം പതിപ്പ്, കീ, കാലാവധി കൂടാതെ, അംഗീകാരം ഉപയോഗിച്ചില്ല എന്ന അവസ്ഥയിൽ. ഓരോ പരാജയവും **വ്യത്യസ്ത കാരണത്തോടെ** നിഷേധിക്കുന്നു, അതുകൊണ്ട് *അധികാരം പഴകിയോ* എന്ന് *നടത്തിയ പ്രവർത്തനം മാറിയോ* എന്ന വ്യത്യാസം മനസിലാകും.


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * ((4 - len(s) % 4) % 4))

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## നിർവ്വചിതമായ പ്രവർത്തനം

അംഗീകാരം നൽകാനുള്ള ഏകകം **കാനോണിക്കൽ പ്രവർത്തന വസ്തു** ആണ് — "റീഫണ്ട് അംഗീകരിക്കുക" പോലുള്ള അസ്പഷ്ടമായ ലേബലല്ല, പക്ഷേ കൃത്യമായ, പൂർണ്ണമായും നിർവ്വചിച്ചിരിക്കുന്ന പ്രവർത്തനമാണ്. മുഴുവൻ വസ്തുവും ഒപ്പിടുകയും അതിൽ നിന്നൊരു ഡൈജസ്റ്റ് ഉത്പാദിപ്പിക്കുകയും ചെയ്യുന്നതാണ് പിന്നീട് മനുഷ്യൻ ഇത് മാത്രമേ അംഗീകരിച്ചതായി തെളിയിക്കാൻ അനുവദിക്കുന്നത്.


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## ഒരു ലിഫ്ഫ്, രണ്ടു അധികാരികൾ

ഓരോ പReceiptയും പാഠത്തിന്റെ ലിഫ്ഫ് ആണ്: ഒരു തളിറച്ച് പayload `type` ഫീൽഡും, കൂടാതെ ഒരെപാട് `signature` വസ്തുവും (`alg`, `sig`, `key_id`) ഇവ ഒപ്പിട്ട ബൈറ്റുകളുടെ ഭാഗമല്ല. `verify_envelope` പങ്കുള്ള ഘടനാപരവും ഒപ്പിടലിന്റെ പരിശോധനയും രണ്ട് Receipt ജീവന്മാരുടേയും സാധാരണമാണ്; അത് `signature.key_id` ഉപയോഗിച്ച് ഏത് **പിന്നഡ് കീ രജിസ്ട്രിക്ക്** ബന്ധിപ്പിക്കുന്നു എന്നതാണ് അധികാരികളെ വേർതിരിക്കുന്നത്:

- **അപ്രൂവൽ റസീറ്റ്** (`human.approval.v1`) — പേരുള്ള അംഗീകരകന്, പൂർണ്ണമായ കാനോണിക്കൽ ആക്ഷൻ **മറ്റും അതിന്റെ ഡൈജസ്റ്റ്**, `policy_version`, ഇറക്കുമതി + കാലാവധി ടൈംസ്റാമ്പുകൾ. ഒറ്റത്തവണ ഉപയോഗം ചെയിൻ നിലയിൽ ട്രാക്ക് ചെയ്യപ്പെടുന്നു.
- **ആക്ഷൻ റസീറ്റ്** (`agent.action.v1`) — ഏജന്റ് തിരിച്ചറിയൽ, `run_id`, അതേ കാനോണിക്കൽ ആക്ഷൻ **ഡൈജസ്റ്റ്**, നിർവഹണഫലവും ടൈംസ്റാമ്പും, കൂടാതെ `parent_approval_ref`: അംഗീകാരം നൽകിയ `receipt_hash`, പാഠത്തിന്റെ ചെയിന്തിലര് ഉള്ള `previous_receipt_hash` എന്ന സമായന്ത്രം പോലെ.

ഒന്നിച്ച് ഉള്ള `action_digest` ഫീൽഡ് ബന്ധത്തിന്റെ വിചാരമാണ്. `key_id` ഒപ്പിടൽ വസ്തുവിൽ ഒരു ലുക്കപ്പ് സൂചനയായി മാത്രമേ നിലനിൽക്കൂ: അത് വേറെPinned കീയെ ലക്ഷ്യം വേർതിരിക്കുകയാണെങ്കിൽ ഒപ്പിടൽ പരിശോധന പരാജയപ്പെടും, അതിനാൽ അത് എന്തിനും ഹിതം നൽകുന്നില്ല.


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the canonical JCS
    bytes directly; the signature object is NOT part of the signed bytes."""
    canonical = canonicalize(payload)
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(canonical).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    pub = trusted_keys.get(sig_obj.get("key_id"))
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized and passed directly to Ed25519.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        canonical = canonicalize(payload)
        VerifyKey(b64url_decode(pub)).verify(canonical, b64url_decode(sig_obj.get("sig") or ""))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## `verify_chain`: ബന്ധിപ്പിക്കല്‍ യാഥാര്‍ത്ഥ്യത്തില്‍ തീരുമാനിക്കുന്ന സ്ഥലം

`verify_chain` രണ്ട് ഒപ്പ് പരിശോധനകളുടെ സൗകര്യവല്‍ക്കരണവുമല്ല. ഇത് ഏകദേശം ഒരിടമാണ്, പങ്കുവെച്ച കാനോണിക്കല്‍ `action_digest`, നയ/കീ/കാലഹരണപ്പെട്ട **പുതിയത്വം** അംഗീകാരം, കൂടാതെ അംഗീകാരം **ഒരിക്കൽ മാത്രം ഉപയോഗം** തമ്മില്‍ ചേര്‍ത്ത് പരിശോധിക്കുക, *ഇപ്പോഴത്തെ* നടപ്പിലാക്കപ്പെടുന്ന പ്രവർത്തനത്തോട് സാന്ദ്രമായി.

ഓരോ പരാജയവും **പ്രത്യേകമായ കാരണ** നല്കിയാണ് നിഷേധിക്കപ്പെടുക, അതുകൊണ്ട് നിഷേധത്തിന്റെ വായനക്കാരന് അറിയാം എങ്കില്‍ അധികാരം പഴകിയതാണോ (നയം മാറി, കീ റൊട്ടേറ്റ് ചെയ്തു, അംഗീകാരം കാലഹരണപ്പെട്ടു, അംഗീകാരം ഉപയോഗിച്ചു) അല്ലെങ്കില്‍ ഇപ്പോഴും സാധുവായ അംഗീകാരവുമായി ബന്ധപ്പെട്ട നടപ്പിലാക്കിയ പ്രവർത്തനം മാറ്റപ്പെട്ടുവെന്ന് (ഡൈജസ്റ്റ് ബദല്‍).


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, executed_at, sk=agent_sk, key_id="agent-key-1"):
    executed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": executed_action,
        "action_digest": sha256_canonical(executed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "performed",
        "executed_at": executed_at,
    }
    return sign_receipt(payload, sk, key_id)

_consumed = set()

def verify_chain(action_being_executed, approval, agent_rcpt, now: str):
    """One code path covers both receipt kinds (same envelope), then checks the
    things that only make sense TOGETHER: shared digest, freshness, consumption.
    `now` is an ISO-8601 Zulu timestamp; Zulu strings compare correctly as strings."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the action being executed
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the executed action changed".
    executing_digest = sha256_canonical(action_being_executed)
    if approval.get("action_digest") != executing_digest or approval.get("action") != action_being_executed:
        return (False, "digest substitution: the approval binds a different canonical action than the one being executed")
    if agent_rcpt.get("action_digest") != executing_digest or agent_rcpt.get("action") != action_being_executed:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the one being executed")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before execution")

    # 4. One-time consumption: an approval authorizes ONE execution.
    ref = receipt_hash(approval)
    if ref in _consumed:
        return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, f"approved by {approval['approver_id']}, executed by {agent_rcpt['agent_id']}")

def execute(action, approval, agent_rcpt, now):
    ok, why = verify_chain(action, approval, agent_rcpt, now)
    return (ok, "executed" if ok else why)

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(execute(action, approval, receipt, now="2026-07-08T15:04:07Z"))

(True, 'executed')


## ബൈൻഡിംഗ് പിടിക്കുന്നതെന്ത്

താഴെ കാണുന്ന ഓരോ കേസുകളും **വ്യത്യസ്ഥ കാരണമെന്ന്** **നിർബന്ധിതമായി** പരാജയപ്പെടുന്നു. ആദ്യ ബ്ലോക്ക് ആണ് പരമ്പരാഗത സെറ്റ് (ടാമ്പർ, കഫ്യൂസ്ഡ് ഡെപ്യൂട്ടി, റിപ്ലേ, യാതൊരുവിധത്തിലും ഫോർജറി, തെറ്റായ ഇൻപുട്ട്). രണ്ടാം ബ്ലോക്ക് ആണ് സ്വത്ത് യാഥാർത്ഥ്യമാക്കുന്ന ജോടി, അവകാശപ്പെട്ടതല്ലാതെ:

- **പഴയ അധികാരം** — സിഗ്നേച്ചർ ഇപ്പോഴും സാധുവാണ്, പക്ഷേ നയം പതിപ്പ് മാറിയിരിക്കുന്നു, അപ്രൂവർ കീ പിന്‍ ചെയ്ത രജിസ്ട്രിയില്‍ നിന്ന് തിരികെ എടുത്തു, അല്ലെങ്കിൽ അപ്രൂവല്‍ നടപ്പാക്കുന്നതിന് മുന്‍പ് കാലഹരണപ്പെട്ടു;
- **ഡൈജസ്റ്റ് സബ്സ്റ്റിറ്റ്യൂഷൻ** — ശരിയായ വഴി സൈൻ ചെയ്‌ത ആക്ഷൻ റിസീറ്റ്, যার `parent_approval_ref` ഒരു *യഥാർത്ഥ* അനുമതിയെ സൂചിപ്പിക്കുന്നു, പക്ഷേ ആ അനുമതി ചിട്ടപ്പെടുത്തിയ ആക്ഷൻ ഡൈജസ്റ്റ് നടപ്പിലാക്കുന്ന ആക്ഷനുമായി പൊരുത്തപ്പെടുന്നില്ല.


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the executed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented to execute action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. replay: the approval was consumed by the successful execution above.
print("replay              ->", execute(action, approval, agent_receipt(action, approval, NOW), NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, un-consumed approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved between approval and
#     execution. Authority is decided at execution time, not signing time.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but execution came too late.
expired = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     is executing action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # executing `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the one being executed')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-receipt   -> (Fa

## ഇത് തെളിയിക്കുന്നത് — അല്ലാത്തത്

**തെളിയിക്കുന്നത്:** പേരുള്ള ഒരു മനുഷ്യൻ *ഈ കണക്കാക്കിയ ക്രിയ (പൂർണ്ണ ക്രിയ + ഡൈജസ്റ്റ്, പിനാക്കിയ രജിസ്റ്റ്രായിൽ നിന്നും കണ്ടെത്തിയ കീ ഉപയോഗിച്ച് ഒപ്പിട്ടത്)* അംഗീകരിച്ചു, എജന്റ് *അതിനെല്ലാം തുല്യമായ അംഗീകൃത ക്രിയ (അേകം ഡൈജസ്റ്റ്, ആംഗീകാരത്തിന് `receipt_hash` എന്നതിൽ ബന്ധിപ്പിച്ച റസീത്, പാഠത്തിന്റെ സ്വന്തം ചെറിയ ഘടന)* നടപ്പിലാക്കി — അംഗീകാരം സാധുവായ നയം പതിപ്പ്, കീ, കാലഹരണ വിവരം ഇപ്പോഴും നിലവിലുണ്ടായിരുന്നപ്പോൾ, ഒരിക്കൽ മാത്രമെ സംഭവിച്ചത്. ഏതെങ്കിലും വശം മാറ്റിയാൽ, ചെയിൻ അടഞ്ഞു_FAIL_ ചെയ്യും, നിരാകരണ കാരണം **ഏത്** സ്വഭാവം തകർക്കപ്പെട്ടുവെന്ന് പറയുന്നു: പഴയ അധികാരമോ മാറിയ ക്രിയയോ.

**തെളിയിക്കുന്നത് അല്ല:** അംഗീകാര UI മനുഷ്യന് അവർ ഒപ്പുവച്ചതായി കരുതിയതെന്താണെന്ന് (WYSIWYS സ്വതന്ത്ര പ്രശ്നമാണ്), കീ മാറുന്നതിനു മുൻപ് നിർബന്ധിതമല്ലാതെയോ മോഷ്ടിക്കപ്പെട്ടതാണെങ്കിൽ, അല്ലെങ്കിൽ താഴെയുള്ള ഫലങ്ങൾ ക്രിയയോട് താരതമ്യപ്പെടുത്തിയിരുന്നെങ്കിൽ. ഒപ്പിട്ടത് ≠ അംഗീകാരം: പഴയ നയം, മാറിയ കീ, കാലഹരണപ്പെട്ട വിൻഡോ, അല്ലെങ്കിൽ വ്യത്യസ്തം ഡൈജസ്റ്റ് ഉള്ള സാധുവായ ഒപ്പിടൽ ഇവിടെ ഒന്നും നൽകുന്നില്ല.

ഈ രണ്ട് റസീത് തരവും പാഠത്തിന്റെ എൻവലപ്പ് പങ്കുവെക്കുകയും `verify_chain` കോഡ് പാത പങ്കിടുകയും ചെയ്യുന്നത് ഉദ്ദേശ്യത്തോടെ ആണ്: മെയിൻ നോട്ട്‌ബുക്കിലെ പ്രവർത്തന റസീത് വേണ്ടി നിങ്ങൾ നിർമ്മിച്ച ബൈൻഡിംഗ് മനുഷ്യന്റെ അംഗീകാരം പരിശോധിക്കുന്നതിന് തുല്യ കോഡ് ആണ്. ഒരേ വെരിഫയർ കരാർ, വ്യത്യസ്ത പിനുകളിച്ച അധികാരങ്ങൾ, കാനോൺ ക്രിയ ഡൈജസ്റ്റ് കൊണ്ടു ചേർന്നിരിക്കുന്നു, മറ്റ് ഒന്നുമില്ല.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
